# System Prompt

Requirements:
- Use preferrably Python `3.12.13` venvs.
    - Using older version might not stream events chunk by chunk, instead it might wait for the **entire** response before printing it.
- Add a `.env` file in the same folder as this notebook with a fields:
    - ANTHROPIC_BASE_URL = `<url_here>`
    - ANTHROPIC_AUTH_TOKEN = "`<jwt_token_here>`"

## 0 Libraries
- anthropic
- dotenv

In [1]:
# %pip install anthropic python-dotenv

## 1 Making a request
- API
- Auth and AI Model

In [2]:
# Env vars, Auth and AI model

from dotenv import load_dotenv
from anthropic import Anthropic

# Env vars

load_dotenv()

# Auth and AI model

client = Anthropic()

model = "bedrock/anthropic.claude-4-6-sonnet"

## 2 Multi-Turn conversations
- Helpers

In [3]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

## 3 System prompt - Math tutor
- Chat func with optional system prompt

In [4]:
# New chat function

def chat(
    messages,
    system=None,
):
    # Required
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    # Optional
    if system:
        params["system"] = system
    
    # KW-args
    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
messages = [] 

user_prompt = "How do I solve 5x + 2 = 3 for x?"
add_user_message(messages, user_prompt)

In [6]:
# Without system prompt

answer = chat(messages)
print(answer)

## Solving for x

Starting with the equation:

**5x + 2 = 3**

**Step 1: Subtract 2 from both sides**
$$5x + 2 - 2 = 3 - 2$$
$$5x = 1$$

**Step 2: Divide both sides by 5**
$$\frac{5x}{5} = \frac{1}{5}$$
$$x = \frac{1}{5}$$

## Answer
**x = 1/5** (or 0.2)

You can verify this by plugging it back into the original equation:
5(1/5) + 2 = 1 + 2 = 3 ✓


In [7]:
# With system prompt

system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

answer = chat(messages, system=system)
print(answer)

Let's work through this step by step!

Your goal is to **isolate x** — get x by itself on one side of the equation.

**First, let's look at what's happening to x in the equation:**
`5x + 2 = 3`

There are two operations applied to x:
- Multiplication by 5
- Addition of 2

**Step 1: What should we deal with first?**

Think about this: what would you do to *both sides* of the equation to start removing things from the left side? 

What operation could you use to "undo" the **+ 2**? 🤔


## 4 System Prompt Exercise

In [8]:
user_prompt = "write a Python function that checks a string for duplicate characters."

In [9]:
# No system prompt 

messages = []

add_user_message(
    messages,
    user_prompt,
)

answer = chat(messages)

print(answer)

## Checking a String for Duplicate Characters

Here's a Python function that checks a string for duplicate characters:

```python
def check_duplicates(string: str) -> dict:
    """
    Checks a string for duplicate characters.

    Args:
        string: The input string to check.

    Returns:
        A dictionary containing:
            - 'has_duplicates': bool indicating if duplicates exist
            - 'duplicates': dict of duplicate characters and their counts
    """
    char_count = {}
    for char in string:
        char_count[char] = char_count.get(char, 0) + 1

    duplicates = {char: count for char, count in char_count.items() if count > 1}

    return {
        "has_duplicates": bool(duplicates),
        "duplicates": duplicates,
    }


# Example usage
if __name__ == "__main__":
    test_strings = [
        "hello",
        "world",
        "python",
        "abcdef",
        "mississippi",
    ]

    for string in test_strings:
        result = check_duplicates(string)
  

In [10]:
# Concise system prompt

system_prompt = "You should write concise code with no explanations."

messages = []

add_user_message(
    messages,
    user_prompt,
)

answer = chat(
    messages,
    system=system_prompt,
)

print(answer)

```python
def has_duplicate_characters(s: str) -> bool:
    return len(s) != len(set(s))
```

**Example usage:**

```python
print(has_duplicate_characters("hello"))   # True  (l appears twice)
print(has_duplicate_characters("world"))   # False
print(has_duplicate_characters("python"))  # False
print(has_duplicate_characters("abcabc")) # True
```

If you also want to **find which characters are duplicated**, use this version:

```python
def find_duplicate_characters(s: str) -> list:
    seen = set()
    duplicates = set()
    for char in s:
        if char in seen:
            duplicates.add(char)
        seen.add(char)
    return list(duplicates)
```

**Example usage:**

```python
print(find_duplicate_characters("hello"))   # ['l']
print(find_duplicate_characters("abcabc"))  # ['a', 'b', 'c']
print(find_duplicate_characters("python"))  # []
```
